# IMPLEMENTATION OF GRU ARCHITECTURE ON XJTU DATASET

This notebook demonstrates the training and validation of the GRU architecture across multiple window sizes on the XJTU-SY bearing dataset.

In [ ]:
import os
import glob
import time
import gc
import re
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import RobustScaler

# Matplotlib Publication Standards
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'lines.linewidth': 2.0,
    'grid.alpha': 0.3
})
sns.set_style("whitegrid")



## 1. Global Configurations
Setup paths, experiment variables, and model hyperparameters.

In [ ]:
# Dataset Paths
DATASET_DIR = r"D:\Proyek Dosen\Riset Bearing\XJTU_Modeling_Input_Dataset"
RESULTS_DIR = r"D:\Proyek Dosen\Riset Bearing\GRU_Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Experiment Setup
WINDOW_SIZES = [30, 40, 50, 70]

# Training Hyperparameters
EPOCHS = 2000
PATIENCE = 150
BATCH_SIZE = 128

# GRU Specific Hyperparameters
HEALTH_INDEX_HIDDEN_LAYER_SIZE = 26
HEALTH_INDEX_NUM_LAYERS = 2
HEALTH_INDEX_DROPOUT_RATE = 0.1
HEALTH_INDEX_LEARNING_RATE = 0.02
HEALTH_INDEX_BETAS = (0.9, 0.999)
HEALTH_INDEX_EPSILON = 1e-8

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")





## 2. Definisi Model GRU
Building the Single Model GRU architecture.

In [ ]:
class HealthIndexGRU(nn.Module):
    def __init__(self, input_size, hidden_layer_size=26, num_layers=2, output_size=1, dropout_rate=0.1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_layer_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_seq):
        gru_out, _ = self.gru(input_seq)
        last_time_step_out = gru_out[:, -1, :]
        dropped = self.dropout(last_time_step_out)
        predictions = self.linear(dropped)
        return predictions


## 3. Helper Functions
Metrics, plotting, and evaluation utilities.

In [ ]:
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def relative_prediction_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def health_index_to_rul(hi, max_rul):
    return np.clip(hi, 0, 1) * max_rul

def asymmetric_loss_scoring_function(y_pred, y_true, a=10, b=13):
    if not isinstance(y_pred, torch.Tensor):
        y_pred = torch.tensor(y_pred, dtype=torch.float32)
        y_true = torch.tensor(y_true, dtype=torch.float32)
    diff = y_pred - y_true
    loss = torch.where(
        diff < 0,
        torch.exp(-diff / a) - 1,
        torch.exp(diff / b) - 1
    )
    return float(loss.mean().item())

def plot_health_index_prediction(y_true, y_pred, title, turning_point_minute=None,
                                  time_axis=None, save_path=None):
    """
    Plots expected vs predicted RUL on the absolute-minute X-axis.
    Args:
        time_axis (np.ndarray): Absolute minute values from Original_Minute.
        turning_point_minute (int): Absolute minute for the CUSUM change point line.
    """
    x = time_axis if time_axis is not None else np.arange(len(y_true))
    plt.figure(figsize=(12, 6))
    plt.plot(x, y_true, label='Expected RUL', color='black', alpha=0.8)
    plt.plot(x, y_pred, label='Predicted RUL', color='gold', linestyle='--')
    if turning_point_minute is not None:
        plt.axvline(x=turning_point_minute, color='red', linestyle='-', alpha=0.7,
                    label=f'Turning Point @ Min {turning_point_minute}')
    plt.title(title, fontweight='bold')
    plt.xlabel('Absolute Time (Minutes)')
    plt.ylabel('Remaining Useful Life (RUL)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()




## 4. Training & Evaluation Pipeline
Looping over specified window sizes to train models and evaluate on validation bearings.

In [ ]:
# Summary Storage
summary_metrics = []
# For saving predictions across all experiments into an Excel file
excel_writer_path = os.path.join(RESULTS_DIR, "GRU_Predictions_Summary.xlsx")
excel_predictions = {} 

for ws in WINDOW_SIZES:
    print(f"\n{'='*60}\nStarting Experiment for Window Size: {ws}\n{'='*60}")
    
    ws_res_dir = os.path.join(RESULTS_DIR, f"window_size_{ws}")
    os.makedirs(ws_res_dir, exist_ok=True)
    
    # ----------------------------------------------------
    # A. LOAD TRAINING DATA
    # ----------------------------------------------------
    train_file = os.path.join(DATASET_DIR, f"processed_train_w{ws}.csv")
    if not os.path.exists(train_file):
        print(f"WARNING: Train file {train_file} not found. Skipping...")
        continue
        
    print(f"Loading training data: {train_file}")
    df_train = pd.read_csv(train_file)
    
    # Train on normalized Health Index [0→1] for stable gradient flow
    y_train = df_train['Health_Index'].values
    if np.isnan(y_train).any() or np.isinf(y_train).any():
        y_train = np.nan_to_num(y_train, nan=0.0, posinf=1.0, neginf=0.0)
    
    # Exclude metadata and target columns
    drop_cols = ['Target_RUL', 'Health_Index']
    if 'Bearing_ID' in df_train.columns: drop_cols.append('Bearing_ID')
    if 'Change_Point' in df_train.columns: drop_cols.append('Change_Point')
    if 'Original_Minute' in df_train.columns: drop_cols.append('Original_Minute')
    X_train_flat = df_train.drop(columns=drop_cols).values

    # Purge Inf/NaN before scaling (prevents MinMaxScaler 1.79e308 trap)
    X_train_flat = np.nan_to_num(X_train_flat, nan=0.0, posinf=0.0, neginf=0.0)

    # RobustScaler: immune to extreme vibration spikes    num_samples = X_train_flat.shape[0]
    num_features = X_train_flat.shape[1] // ws
    X_train_3d_raw = X_train_flat.reshape(num_samples, num_features, ws).transpose(0, 2, 1)
    X_train_3d_2d = X_train_3d_raw.reshape(-1, num_features)

    feature_scaler = RobustScaler()

    X_train_3d_scaled_2d = feature_scaler.fit_transform(X_train_3d_2d)

    X_train_3d = X_train_3d_scaled_2d.reshape(num_samples, ws, num_features)
    
    X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32).to(DEVICE)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(DEVICE)
    
    # Free memory
    del df_train, X_train_flat
    gc.collect()
    
    # ----------------------------------------------------
    # B. INITIALIZE MODEL & TRAIN (GRU)
    # ----------------------------------------------------
    hi_model = HealthIndexGRU(
        input_size=num_features, 
        hidden_layer_size=HEALTH_INDEX_HIDDEN_LAYER_SIZE, 
        num_layers=HEALTH_INDEX_NUM_LAYERS, 
        dropout_rate=HEALTH_INDEX_DROPOUT_RATE
    ).to(DEVICE)
    
    hi_optimizer = optim.Adam(
        hi_model.parameters(), 
        lr=HEALTH_INDEX_LEARNING_RATE, 
        betas=HEALTH_INDEX_BETAS, 
        eps=HEALTH_INDEX_EPSILON
    )
    
    print("Training Model...")
    hi_model.train()
    
    dataset = TensorDataset(X_train_tensor, y_train_tensor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    best_hi_loss = float('inf')
    wait = 0
    best_hi_model_state = None
    hi_loss_history = []
    
    start_time = time.time()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        
        for batch_x, batch_y in dataloader:
            hi_optimizer.zero_grad()
            y_pred = hi_model(batch_x)
            
            # Using custom asymmetric loss calculation on tensors
            diff = y_pred - batch_y
            loss = torch.where(
                diff < 0,
                torch.exp(-diff / 10) - 1,
                torch.exp(diff / 13) - 1
            ).mean()
            
            loss.backward()
            hi_optimizer.step()
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(dataloader)
        hi_loss_history.append(avg_loss)
        
        # Determine improvement based on training loss (mirroring reference logic)
        model_improved = False
        if avg_loss < best_hi_loss:
            best_hi_loss = avg_loss
            best_hi_model_state = copy.deepcopy(hi_model.state_dict())
            model_improved = True
            
        if model_improved:
            wait = 0
        else:
            wait += 1
            
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {(epoch+1)}/{EPOCHS} | HI Loss: {avg_loss:.6f} | Patience: {wait}/{PATIENCE}")
            
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement in {PATIENCE} epochs)")
            break
            
    print(f"Training completed. Best HI Loss: {best_hi_loss:.6f}")
    
    # Load best weights
    hi_model.load_state_dict(best_hi_model_state)
    torch.save(hi_model.state_dict(), os.path.join(ws_res_dir, "best_gru_model.pth"))
    
    # Plot Training Loss
    plt.figure(figsize=(10, 5))
    plt.plot(hi_loss_history, label='Train Loss (Asymmetric Error)', color='red')
    plt.title(f'Training Loss History (Window Size = {ws})')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ws_res_dir, "training_loss_history.png"), dpi=300)
    plt.close()
    
    # Free Memory
    del X_train_tensor, y_train_tensor, dataset, dataloader
    torch.cuda.empty_cache()
    gc.collect()
    
    # ----------------------------------------------------
    # C. EVALUATION ON VALIDATION BEARINGS
    # ----------------------------------------------------
    # Load metadata for lifespan & change point reference
    metadata_file = os.path.join(DATASET_DIR, "bearing_metadata.csv")
    if os.path.exists(metadata_file):
        metadata_df = pd.read_csv(metadata_file)
    else:
        print("[WARN] bearing_metadata.csv not found – falling back to per-bearing estimates.")
        metadata_df = pd.DataFrame(columns=['Bearing_ID', 'Total_Lifespan', 'Change_Point'])

    val_files = glob.glob(os.path.join(DATASET_DIR, f"processed_val_*_w{ws}.csv"))
    
    ws_predictions_df = pd.DataFrame()
    ws_metrics = []
    
    hi_model.eval()
    with torch.no_grad():
        for v_file in val_files:
            b_match = re.search(r"processed_val_(.*)_w\d+\.csv", os.path.basename(v_file))
            b_id = b_match.group(1) if b_match else 'Unknown'
            print(f"Evaluating on {b_id} (Window={ws})...")
            
            df_val = pd.read_csv(v_file)
            y_val_true_hi = df_val['Health_Index'].values
            y_val_true_rul = df_val['Target_RUL'].values

            # Load metadata for this bearing (Total Lifespan for RUL reconstruction)
            b_meta = metadata_df[metadata_df['Bearing_ID'] == b_id]
            if not b_meta.empty:
                b_tf = float(b_meta['Total_Lifespan'].iloc[0])
                b_tcp = int(b_meta['Change_Point'].iloc[0])
            else:
                b_tf = float(df_val['Target_RUL'].max())
                b_tcp = 0

            val_drop_cols = ['Target_RUL', 'Health_Index']
            if 'Bearing_ID' in df_val.columns: val_drop_cols.append('Bearing_ID')
            if 'Change_Point' in df_val.columns: val_drop_cols.append('Change_Point')
            if 'Original_Minute' in df_val.columns: val_drop_cols.append('Original_Minute')
            X_val_flat = df_val.drop(columns=val_drop_cols).values
            X_val_flat = np.nan_to_num(X_val_flat, nan=0.0, posinf=0.0, neginf=0.0)            ns = X_val_flat.shape[0]
            nf = X_val_flat.shape[1] // ws
            X_val_3d_raw = X_val_flat.reshape(ns, nf, ws).transpose(0, 2, 1)
            X_val_3d_2d = X_val_3d_raw.reshape(-1, nf)

            X_val_3d_scaled_2d = feature_scaler.transform(X_val_3d_2d)

            X_val_3d = X_val_3d_scaled_2d.reshape(ns, ws, nf)
            
            X_val_tensor = torch.tensor(X_val_3d, dtype=torch.float32).to(DEVICE)
            
            # Predict HI → reconstruct RUL
            y_val_pred_hi = hi_model(X_val_tensor).cpu().numpy().flatten()
            y_val_pred_hi = np.nan_to_num(y_val_pred_hi, nan=0.0, posinf=0.0, neginf=0.0)
            b_max_rul = b_tf - b_tcp
            y_val_pred_rul = health_index_to_rul(y_val_pred_hi, b_max_rul)

            # Metrics (RUL space)
            rmse_val = rmse_score(y_val_true_rul, y_val_pred_rul)
            mae_val = mean_absolute_error(y_val_true_rul, y_val_pred_rul)
            r2_val = r2_score(y_val_true_rul, y_val_pred_rul)
            rpe_val = relative_prediction_error(y_val_true_rul, y_val_pred_rul)
            asym_loss = asymmetric_loss_scoring_function(y_val_pred_rul, y_val_true_rul)
            
            ws_metrics.append({
                'Bearing_ID': b_id,
                'Window_Size': ws,
                'RMSE': rmse_val,
                'MAE': mae_val,
                'R2': r2_val,
                'RPE(%)': rpe_val,
                'Asymmetric_Loss': asym_loss
            })
            
            # Use Original_Minute as chronological X-axis if available
            if 'Original_Minute' in df_val.columns:
                time_steps = df_val['Original_Minute'].values
            else:
                time_steps = np.arange(len(y_val_true_hi))

            # DataFrame
            b_pred_df = pd.DataFrame({
                'Bearing_ID': b_id,
                'Time_Step': time_steps,
                'Expected_Health_Index': y_val_true_hi,
                'Predicted_Health_Index': y_val_pred_hi,
                'Expected_Remaining_RUL': y_val_true_rul,
                'Predicted_Remaining_RUL': y_val_pred_rul
            })
            ws_predictions_df = pd.concat([ws_predictions_df, b_pred_df], ignore_index=True)
            
            # Plot on absolute minute axis
            plot_title = f"{b_id} RUL Prediction (GRU - WS={ws})"
            plot_save_path = os.path.join(ws_res_dir, f"prediction_plot_{b_id}.png")
            plot_health_index_prediction(y_val_true_rul, y_val_pred_rul, plot_title, b_tcp,
                                         time_axis=time_steps, save_path=plot_save_path)
            
            del df_val, X_val_flat, X_val_3d, X_val_tensor
            gc.collect()
            
    # Save WS Predictions
    ws_predictions_df.to_csv(os.path.join(ws_res_dir, f"predictions_ws_{ws}.csv"), index=False)
    excel_predictions[f"WS_{ws}"] = ws_predictions_df
    
    # Store metrics for Window Size
    if len(ws_metrics) > 0:
        ws_metrics_df = pd.DataFrame(ws_metrics)
        ws_metrics_df.to_csv(os.path.join(ws_res_dir, f"metrics_ws_{ws}.csv"), index=False)
        mean_metrics = ws_metrics_df.mean(numeric_only=True)
        
        summary_metrics.append({
            'Window_Size': ws,
            'Mean_RMSE': mean_metrics['RMSE'],
            'Mean_MAE': mean_metrics['MAE'],
            'Mean_R2': mean_metrics['R2'],
            'Mean_RPE(%)': mean_metrics['RPE(%)'],
            'Mean_Asymmetric_Loss': mean_metrics['Asymmetric_Loss']
        })

# Export Combined Predictions
print(f"\nSaving all predictions to summary Excel: {excel_writer_path}")
with pd.ExcelWriter(excel_writer_path) as writer:
    for sheet_name, df in excel_predictions.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
print("All Experiments Completed Succesfully!")





## 5. Final Window Size Optimization Analysis
Comparing validating mean MSE/RMSE across the 4 window size variants to deduce the optimal length.

In [ ]:
summary_df = pd.DataFrame(summary_metrics)

if not summary_df.empty:
    display(summary_df)
    summary_df.to_csv(os.path.join(RESULTS_DIR, "GRU_WindowSize_Comparison.csv"), index=False)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary_df['Window_Size'], summary_df['Mean_RMSE'], marker='o', linestyle='-', color='purple', linewidth=2.5, markersize=8)
    
    # Annotate points
    for i, row in summary_df.iterrows():
        plt.annotate(f"{row['Mean_RMSE']:.4f}", 
                     (row['Window_Size'], row['Mean_RMSE']),
                     textcoords="offset points", 
                     xytext=(0,10), 
                     ha='center')
                     
    plt.title('Mean Validation RMSE vs Window Size (GRU)', fontweight='bold', pad=15)
    plt.xlabel('Window Size')
    plt.ylabel('Mean Validation RMSE')
    plt.xticks(summary_df['Window_Size'].values)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "optim_gru_rmse_vs_windowsize.png"), dpi=300)
    plt.show()
    
    optimal_ws = summary_df.loc[summary_df['Mean_RMSE'].idxmin()]['Window_Size']
    print(f"\n>>> Based on Mean Validation RMSE, the Optimal Window Size is: {int(optimal_ws)} <<<")
else:
    print("No summary metrics available to analyze.")
